In [22]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# -------- paths  --------
IMG_EMB_PATH = Path("embeddings/image_embeddings.npy")
KW_EMB_PATHS = Path("embeddings/keywords_embeddings.npy")
IMG_INDEX_CSV = Path("csv_files/classified_bucket_with_embeddings.csv")
KW_INDEX_JSON = Path("json_files/keywords_index.json")

# -------- load embeddings --------
from pathlib import Path
I = np.load(IMG_EMB_PATH).astype(np.float32)   # (N_images, 512)

KW_EMB_PATH = Path("embeddings/keywords_embeddings.npy")  # or "keywords_beddings.npy"
if not KW_EMB_PATH.exists():
    KW_EMB_PATH = Path("embeddings/keywords_embeddings.npy")
if not KW_EMB_PATH.exists():
    raise FileNotFoundError("Could not find keywords embeddings .npy (tried keywords_embeddings.npy, keywords_beddings.npy).")

K = np.load(KW_EMB_PATH).astype(np.float32)    # (153, 512)
print("Loaded keyword embeddings from:", KW_EMB_PATH)

# sanity check
assert I.ndim == 2 and K.ndim == 2 and I.shape[1] == K.shape[1] == 512, f"Unexpected shapes: {I.shape}, {K.shape}"


# -------- L2 normalize so dot = cosine --------
def l2_normalize(mat, axis=1, eps=1e-12):
    norms = np.linalg.norm(mat, axis=axis, keepdims=True)
    return mat / np.clip(norms, eps, None)

I_u = l2_normalize(I)          # (N, 512)
K_u = l2_normalize(K)          # (153, 512)

# -------- compute cosine similarities --------
# S[i, j] = cosine(image i, keyword j)
S = I_u @ K_u.T                # (N, 153), float32

# -------- load indexes for human-readable labels --------
# Image index CSV: we'll try to pick a sensible label column.
img_df = pd.read_csv(IMG_INDEX_CSV)

# Try common label columns; fall back to row index.
candidate_cols = ["filename", "image_path", "image", "image_id", "id", "name", "label", "location", "title"]
img_label_col = next((c for c in candidate_cols if c in img_df.columns), None)
if img_label_col is None:
    img_df["image_label"] = img_df.index.astype(str)
    img_label_col = "image_label"

with open(KW_INDEX_JSON, "r") as f:
    kw_index = json.load(f)
# Expect either {int_str: "phrase"} or a list; normalize to list
if isinstance(kw_index, dict):
    # sort by int key
    kw_labels = [kw_index[str(i)] if str(i) in kw_index else kw_index[i] for i in sorted(map(int, kw_index.keys()))]
elif isinstance(kw_index, list):
    kw_labels = kw_index
else:
    raise ValueError("keywords_index.json should be a dict or list")

assert len(kw_labels) == K.shape[0], "Keyword index length doesn't match embeddings."

# -------- Top-k keywords per image --------
K_TOP = 10
# argpartition is memory-friendly
topk_idx = np.argpartition(-S, K_TOP-1, axis=1)[:, :K_TOP]  # unsorted top-k
# sort the top-k slice
row_indices = np.arange(S.shape[0])[:, None]
topk_scores = S[row_indices, topk_idx]
order = np.argsort(-topk_scores, axis=1)
topk_idx_sorted = topk_idx[row_indices, order]
topk_scores_sorted = topk_scores[row_indices, order]

# Build a tidy dataframe with top-k keywords per image
def pack_pairs(indices_row, scores_row):
    return [(kw_labels[j], float(scores_row[i])) for i, j in enumerate(indices_row)]

topk_pairs = [pack_pairs(topk_idx_sorted[i], topk_scores_sorted[i]) for i in range(S.shape[0])]
img_topk_df = pd.DataFrame({
    "image_row": np.arange(S.shape[0]),
    "image_label": img_df[img_label_col].values[:S.shape[0]],
    f"top{K_TOP}_keyword_matches": topk_pairs
})

# Also provide a flat table (one row per (image, keyword) in top-k) for easier filtering
flat_rows = []
for i in range(S.shape[0]):
    label = img_df.iloc[i][img_label_col]
    for (kw, sc) in topk_pairs[i]:
        flat_rows.append({"image_row": i, "image_label": label, "keyword": kw, "score": sc})
img_kw_topk_flat = pd.DataFrame(flat_rows)

# -------- Top images per keyword (reverse lookup) --------
I_TOP = 10
topi_idx = np.argpartition(-S, I_TOP-1, axis=0)[:I_TOP, :]  # top I_TOP rows per column (unsorted)
# sort within each column
topi_scores = S[topi_idx, np.arange(S.shape[1])]
order_cols = np.argsort(-topi_scores, axis=0)
topi_idx_sorted = topi_idx[order_cols, np.arange(S.shape[1])]
topi_scores_sorted = topi_scores[order_cols, np.arange(S.shape[1])]

kw_topi = []
for j in range(S.shape[1]):
    rows = []
    for r in range(I_TOP):
        i = topi_idx_sorted[r, j]
        rows.append(
            {"keyword": kw_labels[j],
             "image_row": int(i),
             "image_label": img_df.iloc[i][img_label_col],
             "score": float(topi_scores_sorted[r, j])}
        )
    kw_topi.extend(rows)
kw_top_images_df = pd.DataFrame(kw_topi)

# -------- Diagnostics to judge label suitability --------
# Per image: top-1 score, margin top1-top2, entropy-like dispersion
top1 = topk_scores_sorted[:, 0]
top2 = topk_scores_sorted[:, 1]
margin = top1 - top2

# Softmax over keywords (temperature optional)
def softmax(x, T=0.07):  # lower T -> sharper distribution; 0.07 is a common CLIP temp
    z = (x / T)
    z -= z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

P = softmax(S, T=0.07)
entropy = -np.sum(P * np.log(np.clip(P, 1e-12, None)), axis=1)

diagnostics_df = pd.DataFrame({
    "image_row": np.arange(S.shape[0]),
    "image_label": img_df[img_label_col].values[:S.shape[0]],
    "top1_score": top1,
    "top1_minus_top2_margin": margin,
    "entropy": entropy
})

# Keyword coverage statistics: fraction of images where keyword appears in top-k
coverage = (topk_idx_sorted.flatten()).reshape(S.shape[0], K_TOP)
kw_counts = np.bincount(coverage.flatten(), minlength=S.shape[1])
kw_coverage_df = pd.DataFrame({
    "keyword": kw_labels,
    "in_topk_count": kw_counts,
    "in_topk_pct": kw_counts / S.shape[0]
}).sort_values("in_topk_count", ascending=False)

# -------- Save outputs  --------
#img_kw_topk_flat.to_parquet("image_keyword_topk_flat.parquet", index=False)
#kw_top_images_df.to_parquet("keyword_top_images.parquet", index=False)
#diagnostics_df.to_parquet("similarity_diagnostics.parquet", index=False)
#kw_coverage_df.to_parquet("keyword_coverage.parquet", index=False)

img_kw_topk_flat.to_csv("image_keyword_topk_flat.csv", index=False, encoding="utf-8")
kw_top_images_df.to_csv("keyword_top_images.csv", index=False, encoding="utf-8")
diagnostics_df.to_csv("similarity_diagnostics.csv", index=False, encoding="utf-8")
kw_coverage_df.to_csv("keyword_coverage.csv", index=False, encoding="utf-8")

top1_kw_idx = topk_idx_sorted[:, 0]
top1_kw = [kw_labels[j] for j in top1_kw_idx]
top1_score = topk_scores_sorted[:, 0]
top2_score = topk_scores_sorted[:, 1]
image_top1_df = pd.DataFrame({
    "image_row": np.arange(S.shape[0]),
    "image_label": img_df[img_label_col].values[:S.shape[0]],
    "top_keyword": top1_kw,
    "top_score": top1_score,
    "margin_top1_top2": top1_score - top2_score,
    "entropy": diagnostics_df["entropy"].values
})


print("Done. Files written:\n",
      "- image_topk_keywords.parquet\n",
      "- image_keyword_topk_flat.parquet\n",
      "- keyword_top_images.parquet\n",
      "- similarity_diagnostics.parquet\n",
      "- keyword_coverage.parquet")


Loaded keyword embeddings from: embeddings/keywords_embeddings.npy
Done. Files written:
 - image_topk_keywords.parquet
 - image_keyword_topk_flat.parquet
 - keyword_top_images.parquet
 - similarity_diagnostics.parquet
 - keyword_coverage.parquet


In [21]:
image_top1_df.head()

,image_row,image_label,top_keyword,top_score,margin_top1_top2,entropy
0,0,00bcce0f.jpg,"{'column': 'Vibes', 'row_idx': 41, 'text': 'a ...",0.272206,0.004272,4.974130
1,1,01146212.jpg,"{'column': 'Activities', 'row_idx': 14, 'text'...",0.278062,0.000172,4.958305
2,2,018b51ef.jpg,"{'column': 'Vibes', 'row_idx': 41, 'text': 'a ...",0.279630,0.003670,4.939931
3,3,02.jpg,"{'column': 'Vibes', 'row_idx': 41, 'text': 'a ...",0.263301,0.001563,4.975334
4,4,025f1dd8.jpg,"{'column': 'Vibes', 'row_idx': 22, 'text': 'a ...",0.278989,0.011211,4.975454


In [10]:
# listing all 153 keyword-phrases and list how many images have a significant similarity with that keyword phrase

kw_display = [k.get("text", str(k)) if isinstance(k, dict) else str(k) for k in kw_labels]

def keyword_significance_table(S, kw_display, t=0.32, topk=None):
    import numpy as np, pandas as pd
    N, M = S.shape
    mask = S >= t
    count = mask.sum(axis=0)
    pct = count / N
    extra = {}
    if topk is not None:
        topk_idx = np.argpartition(-S, topk-1, axis=1)[:, :topk]
        in_topk = np.zeros_like(S, dtype=bool)
        in_topk[np.arange(N)[:,None], topk_idx] = True
        conf = mask & in_topk
        extra["count_ge_t_and_topk"] = conf.sum(axis=0)
        extra["pct_ge_t_and_topk"] = extra["count_ge_t_and_topk"] / N
    return (pd.DataFrame({
        "keyword": kw_display,
        "count_ge_t": count,
        "pct_ge_t": pct,
        "median_score": np.median(S, axis=0),
        "p95_score": np.quantile(S, 0.95, axis=0),
        "threshold": t,
        **extra
    }).sort_values("count_ge_t", ascending=False).reset_index(drop=True))

t = 0.20
tbl = keyword_significance_table(S, kw_display, t=t, topk=10)  # add topk if you want stricter counting
tbl.to_csv(f"keyword_significance_t{int(t*100):02d}.csv", index=False)
tbl.head()


,keyword,count_ge_t,pct_ge_t,median_score,p95_score,threshold,count_ge_t_and_topk,pct_ge_t_and_topk
0,a travel scene that feels relaxing,100,1.0,0.240723,0.256248,0.2,0,0.00
1,a travel scene that feels cosmopolitan,100,1.0,0.238972,0.261062,0.2,6,0.06
2,a travel scene that feels edgy,100,1.0,0.226609,0.247002,0.2,0,0.00
3,a travel scene that feels no-frills,100,1.0,0.246330,0.265716,0.2,19,0.19
4,a travel scene that feels backpacker,100,1.0,0.246652,0.262418,0.2,23,0.23


In [11]:
print("I:", I.shape)   # expect (20888, 512)
print("K:", K.shape)   # expect (153, 512)
print("S:", S.shape)   # should be (20888, 153)

I: (100, 512)
K: (153, 512)
S: (100, 153)


In [13]:
images_df = pd.read_csv("csv_files/classified_bucket_with_embeddings.csv")

In [15]:
images_df.shape

(20888, 3)

In [17]:
array = np.load("embeddings/image_embeddings.npy")

In [19]:
array.shape

(100, 512)